# Heart Disease Prediction — Data Preparation & Visualization
**CSE570 Skill Based Assignment**

Covers rubric Section 2(a) — Data Cleaning and Visualization (5 marks).


Prerequisite: none. This is the first notebook.

## 1. Setup and Data Loading
Data source: UCI Machine Learning Repository — Heart Disease (Cleveland, 303 records).
The raw file `processed.cleveland.data` is comma-separated with no header; missing values are encoded as `?`.

If the download fails, fetch the file manually from https://archive.ics.uci.edu/dataset/45/heart+disease and place it next to this notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)

# Column names for the Cleveland dataset (14 attributes)
cols = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs',
        'restecg', 'thalach', 'exang', 'oldpeak', 'slope',
        'ca', 'thal', 'num']

URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data'
# Try the UCI URL first; fall back to a local copy if the download fails
import os
LOCAL = 'processed.cleveland.data'
try:
    df = pd.read_csv(URL, names=cols, na_values='?')
except Exception as e:
    print('URL download failed -> using local file:', e)
    df = pd.read_csv(LOCAL, names=cols, na_values='?')

print(df.shape)   # Expected: (303, 14)
df.head()

## 2. Data Cleaning


### 2.1 Missing-value summary & treatment
Only `ca` and `thal` contain missing values. Since only ~2% of rows are affected, we drop them — simple and safe for a 303-row dataset.

In [ ]:
print(df.isna().sum())
print(f'\nRows with any missing value: {df.isna().any(axis=1).sum()} ({df.isna().any(axis=1).mean()*100:.1f}%)')

df = df.dropna().reset_index(drop=True)
print(f'\nShape after dropping missing rows: {df.shape}')

### 2.2 Duplicate handling


In [ ]:
dups = df.duplicated().sum()
print(f'Duplicate rows: {dups}')
if dups:
    df = df.drop_duplicates().reset_index(drop=True)

### 2.3 Datatype correction
`ca` and `thal` arrive as floats only because of NaN; they are categorical codes and are cast to the correct dtype.

In [ ]:
cat_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal', 'num']
df[cat_cols] = df[cat_cols].astype(int)

for c in ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']:
    df[c] = df[c].astype('category')

df.info()

### 2.4 Target binarization
`num` = 0 means no disease; 1-4 are increasing severity levels. Following standard practice we binarize: 0 = absence, 1 = presence.

In [ ]:
df['target'] = (df['num'] > 0).astype(int)
df = df.drop(columns='num')

num_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
cat_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']

print(df['target'].value_counts())
print(df['target'].value_counts(normalize=True).round(3))

### 2.5 Outlier inspection
Numeric features: age, trestbps, chol, thalach, oldpeak. We inspect with boxplots and the IQR rule. Extreme trestbps/chol values are physiologically possible, so we keep them but report them.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, c in zip(axes, num_cols):
    df[c].plot.box(ax=ax)
    ax.set_title(c)
plt.tight_layout()
plt.savefig('outliers_boxplot.png', dpi=150)
plt.show()

for c in num_cols:
    q1, q3 = df[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    n_out = ((df[c] < q1 - 1.5*iqr) | (df[c] > q3 + 1.5*iqr)).sum()
    print(f'{c}: {n_out} outliers ({n_out/len(df)*100:.1f}%)')

## 3. Train-Test Split

**Why first?** Scaling and encoding must learn their parameters (means, stds, categories) from the training data only. Splitting first and fitting transformers on the train set prevents **data leakage** between the training and test sets.

We use a stratified 80/20 split so both classes keep the same 54/46 proportion in each subset — important because the dataset is small.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns='target')
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print('Train class proportion:\n', y_train.value_counts(normalize=True).round(3))
print('Test class proportion:\n', y_test.value_counts(normalize=True).round(3))

## 4. Preprocessing Pipeline

Built once with `ColumnTransformer` and reused for every model:
- **Numeric features** → `StandardScaler` (feature scaling; needed for Logistic Regression, KNN, SVM)
- **Categorical features** → `OneHotEncoder` (encoding; drop='first' avoids the dummy-variable trap)

Tree-based models (Decision Tree, Random Forest) do not need scaling, but keeping one shared leakage-free pipeline is cleaner and lets us compare all models on identical data.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_cols)
])

# Fit on TRAIN only (no leakage), then transform test
X_train_enc = preprocessor.fit_transform(X_train)
X_test_enc  = preprocessor.transform(X_test)

print(f'Encoded train shape: {X_train_enc.shape}')
print(f'Encoded test shape:  {X_test_enc.shape}')
print('Feature names:', preprocessor.get_feature_names_out()[:8], '...')

## 5. Visualizations — Key Data Characteristics


### 5.1 Target class distribution


In [ ]:
ax = sns.countplot(x=df['target'])
ax.set_xticklabels(['No disease', 'Disease'])
ax.set_xlabel('Target'); ax.set_ylabel('Count')
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x()+p.get_width()/2, p.get_height()),
                ha='center', va='bottom')
plt.tight_layout()
plt.savefig('target_distribution.png', dpi=150)
plt.show()

### 5.2 Distribution of numeric features by target


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()
for i, c in enumerate(num_cols):
    sns.histplot(data=df, x=c, hue='target', kde=True, ax=axes[i], bins=20)
    axes[i].set_title(f'{c} by target')
axes[5].axis('off')
plt.tight_layout()
plt.savefig('numeric_by_target.png', dpi=150)
plt.show()

### 5.3 Categorical features vs. target


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, c in zip(axes.ravel(), cat_cols):
    sns.countplot(data=df, x=c, hue='target', ax=ax)
    ax.set_title(c)
plt.tight_layout()
plt.savefig('categorical_by_target.png', dpi=150)
plt.show()

### 5.4 Correlation heatmap


In [ ]:
corr = df[num_cols + ['target']].corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150)
plt.show()

## 6. Save the cleaned dataset


In [ ]:
df.to_csv('heart_disease_clean.csv', index=False)
print('Saved: heart_disease_clean.csv')
df.describe(include='all').T

---
